# Model Optimization — ONNX Export & Compression Benchmark

This notebook exports the trained model to ONNX and benchmarks:
- **PyTorch FP32** (baseline)
- **ONNX FP32** (graph optimized)
- **ONNX FP16** (GPU optimized)

Compares speed and accuracy (IoU) for each variant on the validation set.

In [ ]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"

if IN_COLAB:
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "onnx", "onnxruntime-gpu"], check=True)
    PROJECT_ROOT = __import__('pathlib').Path(REPO_DIR).resolve()
else:
    PROJECT_ROOT = __import__('pathlib').Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

import warnings
warnings.filterwarnings("ignore")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import torch
import numpy as np
import time
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 1. Load Checkpoint & Export ONNX

In [ ]:
# ====== SET CHECKPOINT PATH ======
CHECKPOINT = ""  # <-- SET THIS
assert CHECKPOINT, "Set CHECKPOINT above"

OUTPUT_DIR = Path("models")
OUTPUT_DIR.mkdir(exist_ok=True)

from road_segmentation.api.optimize import (
    load_model_from_checkpoint,
    export_onnx,
    convert_onnx_fp16,
)

model, config = load_model_from_checkpoint(CHECKPOINT, device="cpu")
image_size = config.get("data", {}).get("image_size", 512)
model_cfg = config.get("model", {})
model_name = f"{model_cfg.get('arch')}_{model_cfg.get('encoder_name')}"

print(f"Model: {model_name} | image_size={image_size}")
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

# Export FP32 ONNX
onnx_fp32_path = OUTPUT_DIR / f"{model_name}.onnx"
export_onnx(model, onnx_fp32_path, image_size=image_size)

# Export FP16 ONNX
onnx_fp16_path = convert_onnx_fp16(onnx_fp32_path)

# Print sizes
for p in [onnx_fp32_path, onnx_fp16_path]:
    print(f"  {p.name}: {p.stat().st_size / 1e6:.1f} MB")

---
## 2. Speed Benchmark

Compare inference latency: PyTorch FP32 vs ONNX FP32 vs ONNX FP16 on GPU.

In [ ]:
import onnxruntime as ort

N_WARMUP = 5
N_RUNS = 30
dummy_np = np.random.randn(1, 3, image_size, image_size).astype(np.float32)
dummy_torch = torch.from_numpy(dummy_np)


def benchmark_pytorch(model, device, n_warmup=N_WARMUP, n_runs=N_RUNS):
    model = model.to(device).eval()
    x = dummy_torch.to(device)
    # Warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    # Timed runs
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(x)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
    return times


def benchmark_onnx(onnx_path, provider, n_warmup=N_WARMUP, n_runs=N_RUNS):
    providers = [provider, "CPUExecutionProvider"] if provider != "CPUExecutionProvider" else ["CPUExecutionProvider"]
    sess = ort.InferenceSession(str(onnx_path), providers=providers)
    input_name = sess.get_inputs()[0].name
    active = sess.get_providers()[0]
    print(f"  Active provider: {active}")
    # Warmup
    for _ in range(n_warmup):
        _ = sess.run(None, {input_name: dummy_np})
    # Timed runs
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = sess.run(None, {input_name: dummy_np})
        times.append((time.perf_counter() - t0) * 1000)
    return times


results = {}

# PyTorch FP32 on GPU
print("PyTorch FP32 (GPU)...")
results["PyTorch FP32 GPU"] = benchmark_pytorch(model, device)

# PyTorch FP16 on GPU (AMP)
print("PyTorch FP16/AMP (GPU)...")
model_gpu = model.to(device).eval()
x_gpu = dummy_torch.to(device)
times_amp = []
with torch.no_grad():
    for _ in range(N_WARMUP):
        with torch.amp.autocast(device_type="cuda"):
            _ = model_gpu(x_gpu)
    torch.cuda.synchronize()
    for _ in range(N_RUNS):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.amp.autocast(device_type="cuda"):
            _ = model_gpu(x_gpu)
        torch.cuda.synchronize()
        times_amp.append((time.perf_counter() - t0) * 1000)
results["PyTorch FP16/AMP GPU"] = times_amp

# ONNX FP32 on GPU
print("ONNX FP32 (GPU)...")
results["ONNX FP32 GPU"] = benchmark_onnx(onnx_fp32_path, "CUDAExecutionProvider")

# ONNX FP16 on GPU
print("ONNX FP16 (GPU)...")
results["ONNX FP16 GPU"] = benchmark_onnx(onnx_fp16_path, "CUDAExecutionProvider")

print("\nDone.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("ggplot")

rows = []
for name, times in results.items():
    arr = np.array(times)
    rows.append({
        "Variant": name,
        "Mean (ms)": arr.mean(),
        "Median (ms)": np.median(arr),
        "p95 (ms)": np.percentile(arr, 95),
        "Std (ms)": arr.std(),
    })

bench_df = pd.DataFrame(rows)
print("Latency Benchmark (single image inference)")
print("=" * 60)
display(bench_df.round(2))

# Speedup relative to PyTorch FP32
baseline = bench_df.iloc[0]["Median (ms)"]
bench_df["Speedup"] = baseline / bench_df["Median (ms)"]

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#2b8cbe", "#7bccc4", "#f0b27a", "#ef6548"]
axes[0].barh(bench_df["Variant"], bench_df["Median (ms)"], color=colors[:len(bench_df)])
axes[0].set_xlabel("Median Latency (ms)")
axes[0].set_title("Inference Latency")
for i, v in enumerate(bench_df["Median (ms)"]):
    axes[0].text(v + 1, i, f"{v:.1f}ms", va="center")

axes[1].barh(bench_df["Variant"], bench_df["Speedup"], color=colors[:len(bench_df)])
axes[1].set_xlabel("Speedup vs PyTorch FP32")
axes[1].set_title("Speedup")
axes[1].axvline(1.0, color="black", linewidth=0.5, linestyle="--")
for i, v in enumerate(bench_df["Speedup"]):
    axes[1].text(v + 0.05, i, f"{v:.2f}x", va="center")

plt.tight_layout()
plt.show()

---
## 3. Accuracy Comparison

Verify that ONNX export and FP16 conversion don't degrade IoU.

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR
from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs
from road_segmentation.data.transforms import get_val_transform
from road_segmentation.data.dataset import RoadSegmentationDataset
from torch.utils.data import DataLoader
from tqdm import tqdm

norm_cfg = config.get("normalization", {})
mean = norm_cfg.get("mean", [0.485, 0.456, 0.406])
std = norm_cfg.get("std", [0.229, 0.224, 0.225])

pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
_, val_pairs = split_pairs(
    pairs,
    val_ratio=config.get("data", {}).get("val_split_ratio", 0.15),
    seed=config.get("data", {}).get("val_split_seed", 42),
)

# Use a subset for speed
N_ACC_SAMPLES = min(200, len(val_pairs))
val_pairs_sub = val_pairs[:N_ACC_SAMPLES]

val_transform = get_val_transform(image_size, mean, std)
val_dataset = RoadSegmentationDataset(val_pairs_sub, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)


def compute_iou_pytorch(model, loader, device):
    model = model.to(device).eval()
    tp, fp, fn = 0, 0, 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="PyTorch"):
            imgs = batch["image"].to(device)
            masks = batch["mask"]
            probs = torch.sigmoid(model(imgs)).cpu()
            preds = (probs >= 0.5).float()
            tp += ((preds == 1) & (masks == 1)).sum().item()
            fp += ((preds == 1) & (masks == 0)).sum().item()
            fn += ((preds == 0) & (masks == 1)).sum().item()
    return tp / max(tp + fp + fn, 1)


def compute_iou_onnx(onnx_path, loader, provider):
    providers = [provider, "CPUExecutionProvider"] if provider != "CPUExecutionProvider" else ["CPUExecutionProvider"]
    sess = ort.InferenceSession(str(onnx_path), providers=providers)
    input_name = sess.get_inputs()[0].name
    tp, fp, fn = 0, 0, 0
    for batch in tqdm(loader, desc=f"ONNX {Path(onnx_path).stem}"):
        imgs = batch["image"].numpy()
        masks = batch["mask"].numpy()
        logits = sess.run(None, {input_name: imgs})[0]
        probs = 1 / (1 + np.exp(-logits))
        preds = (probs >= 0.5).astype(float)
        tp += ((preds == 1) & (masks == 1)).sum()
        fp += ((preds == 1) & (masks == 0)).sum()
        fn += ((preds == 0) & (masks == 1)).sum()
    return tp / max(tp + fp + fn, 1)


print(f"Computing IoU on {N_ACC_SAMPLES} validation samples...\n")

iou_pytorch = compute_iou_pytorch(model, val_loader, device)
iou_onnx_fp32 = compute_iou_onnx(onnx_fp32_path, val_loader, "CUDAExecutionProvider")
iou_onnx_fp16 = compute_iou_onnx(onnx_fp16_path, val_loader, "CUDAExecutionProvider")

acc_rows = [
    {"Variant": "PyTorch FP32", "IoU": iou_pytorch, "Delta vs FP32": 0},
    {"Variant": "ONNX FP32", "IoU": iou_onnx_fp32, "Delta vs FP32": iou_onnx_fp32 - iou_pytorch},
    {"Variant": "ONNX FP16", "IoU": iou_onnx_fp16, "Delta vs FP32": iou_onnx_fp16 - iou_pytorch},
]
acc_df = pd.DataFrame(acc_rows)

print("\nAccuracy Comparison")
print("=" * 50)
display(acc_df.round(6))

if abs(iou_onnx_fp16 - iou_pytorch) < 0.005:
    print("\nFP16 accuracy loss is negligible (<0.5% IoU). Safe for production.")
else:
    print(f"\nFP16 accuracy drop: {abs(iou_onnx_fp16 - iou_pytorch):.4f} IoU. Consider using FP32.")

---
## 4. Summary

In [ ]:
# Merge speed and accuracy
print("Optimization Summary")
print("=" * 70)
print(f"{'Variant':<25} {'Latency (ms)':>12} {'Speedup':>8} {'IoU':>8} {'Size (MB)':>10}")
print("-" * 70)

summary_data = [
    ("PyTorch FP32 GPU", bench_df.iloc[0]["Median (ms)"], 1.0, iou_pytorch, "-"),
    ("PyTorch FP16/AMP GPU", bench_df.iloc[1]["Median (ms)"], bench_df.iloc[1]["Speedup"], iou_pytorch, "-"),
    ("ONNX FP32 GPU", bench_df.iloc[2]["Median (ms)"], bench_df.iloc[2]["Speedup"], iou_onnx_fp32,
     f"{onnx_fp32_path.stat().st_size/1e6:.1f}"),
    ("ONNX FP16 GPU", bench_df.iloc[3]["Median (ms)"], bench_df.iloc[3]["Speedup"], iou_onnx_fp16,
     f"{onnx_fp16_path.stat().st_size/1e6:.1f}"),
]

for name, latency, speedup, iou, size in summary_data:
    print(f"{name:<25} {latency:>10.1f}ms {speedup:>7.2f}x {iou:>7.4f} {size:>10}")

print(f"\nModels saved to: {OUTPUT_DIR}")
for p in OUTPUT_DIR.glob("*.onnx"):
    print(f"  {p.name} ({p.stat().st_size/1e6:.1f} MB)")

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = Path("/content/drive/MyDrive/road_segmentation/models")
    save_dir.mkdir(parents=True, exist_ok=True)
    for p in OUTPUT_DIR.glob("*.onnx"):
        shutil.copy(p, save_dir / p.name)
        print(f"Saved: {save_dir / p.name}")